In [4]:
import dgl
import torch

In [5]:
from dgl import remove_self_loop

In [13]:
g = dgl.graph((torch.tensor([0, 0, 0, 1]), torch.tensor([1, 0, 0, 2])))
g

Graph(num_nodes=3, num_edges=4,
      ndata_schemes={}
      edata_schemes={})

In [21]:
g.directed

AttributeError: 'DGLGraph' object has no attribute 'directed'

In [10]:
g.edges()

(tensor([0, 0, 0, 1]), tensor([1, 0, 0, 2]))

In [14]:
g.all_edges()

(tensor([0, 0, 0, 1]), tensor([1, 0, 0, 2]))

In [18]:
g.in_edges(0)

(tensor([0, 0]), tensor([0, 0]))

In [11]:
remove_self_loop(g).edges()

(tensor([0, 1]), tensor([1, 2]))

In [12]:
remove_self_loop(g)

Graph(num_nodes=3, num_edges=2,
      ndata_schemes={}
      edata_schemes={})

In [26]:
from dgl import remove_self_loop
import numpy as np
def _construct_dgl_graph(
    adjacency_matrix_rows_cols, # dict containing "row_coords" key - node indices of all source nodes and "col_coords" - node of all target nodes respectively
    features: np.ndarray, # [N, d] feature array, where N==number of nodes, d is the initial feature dimension
    targets: np.ndarray, # [1, ] label column
    node_ids: np.ndarray, # unique node ids
    # masks for proper nodes separation on train/val/test subsets during training:
    train_mask: np.ndarray,
    val_mask: np.ndarray,
    test_mask: np.ndarray,
):
    row_coordinates, col_coordinates = (
        adjacency_matrix_rows_cols["row_coords"],
        adjacency_matrix_rows_cols["col_coords"],
    )

    row_coordinates = torch.tensor(row_coordinates).long()
    col_coordinates = torch.tensor(col_coordinates).long()
    graph = dgl.graph(data=(row_coordinates, col_coordinates), idtype=torch.int32, num_nodes=len(node_ids))
    graph.ndata["features"] = torch.tensor(features, dtype=torch.float32)
    graph.ndata["labels"] = torch.tensor(targets, dtype=torch.float32).reshape(-1, 1)

    graph.ndata["train"] = torch.tensor(train_mask, dtype=torch.bool).reshape(-1, 1)
    graph.ndata["val"] = torch.tensor(val_mask, dtype=torch.bool).reshape(-1, 1)
    graph.ndata["test"] = torch.tensor(test_mask, dtype=torch.bool).reshape(-1, 1)

    graph.ndata["ids"] = torch.tensor(node_ids, dtype=torch.long).reshape(-1, 1)

    return graph


adjacency_matrix_rows_cols = {
    "row_coords": np.random.randint(0, 100, size=1000),
    "col_coords": np.random.randint(0, 100, size=1000)
}

features = np.random.randn(100, 10)
targets = np.random.randint(0, 2, size=100)
node_ids = np.arange(100)
train_mask = np.random.choice([0, 1], size=(100,), p=[1./2, 1./2])
val_mask = np.random.choice([0, 1], size=(100,), p=[1./2, 1./2])
test_mask = np.random.choice([0, 1], size=(100,), p=[1./2, 1./2])

# Create the graph
graph = _construct_dgl_graph(adjacency_matrix_rows_cols, features, targets, node_ids, train_mask, val_mask, test_mask)

# Remove self-loops
graph_1 = remove_self_loop(graph)

In [27]:
graph

Graph(num_nodes=100, num_edges=1000,
      ndata_schemes={'features': Scheme(shape=(10,), dtype=torch.float32), 'labels': Scheme(shape=(1,), dtype=torch.float32), 'train': Scheme(shape=(1,), dtype=torch.bool), 'val': Scheme(shape=(1,), dtype=torch.bool), 'test': Scheme(shape=(1,), dtype=torch.bool), 'ids': Scheme(shape=(1,), dtype=torch.int64)}
      edata_schemes={})

In [28]:
graph_1

Graph(num_nodes=100, num_edges=983,
      ndata_schemes={'features': Scheme(shape=(10,), dtype=torch.float32), 'labels': Scheme(shape=(1,), dtype=torch.float32), 'train': Scheme(shape=(1,), dtype=torch.bool), 'val': Scheme(shape=(1,), dtype=torch.bool), 'test': Scheme(shape=(1,), dtype=torch.bool), 'ids': Scheme(shape=(1,), dtype=torch.int64)}
      edata_schemes={})